# Raven Input File Creation

In [111]:
import os
import sys
import fileinput
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint
from datetime import datetime

sys.path.append("..")
from src.py.proj_conf import *

In [5]:
# Prompt user to input model name
model_name = input("Model Name: ")

## Import CLRH as Geodataframe

In [6]:
# Read clrh shapefile
mc_clrh_gdf = gpd.read_file(DATA_RAW + "CLRH_RoutingProduct/05OE006/finalcat_info_v1-0.shp")

# Project clrh gdf to WGS 84
mc_clrh_gdf_project = mc_clrh_gdf.to_crs(4326)

In [7]:
mc_clrh_gdf_project

,SubId,DowSubId,RivSlope,RivLength,BasSlope,BasAspect,BasArea,BkfWidth,BkfDepth,Lake_Cat,...,DA_Chn_L,DA_Slope,DA_Chn_Slp,outletLat,outletLng,k,c,SSDA_NM,SDA_NM,geometry
0,310013151,310015352,0.001359,1472.132034,0.139288,101.631184,1020600.0,3.598586,0.178092,0.0,...,12578.376618,0.083886,0.000743,49.408374,-96.609506,0.004507,0.985797,Nelson,05OS,"POLYGON ((-96.6018 49.40315, -96.60346 49.4031..."
1,310013152,310015352,0.000777,3362.132034,0.341033,113.693816,5556600.0,1.557162,0.107737,0.0,...,3362.132034,0.307192,0.000777,49.408101,-96.609915,0.004507,0.985797,Nelson,05OS,"POLYGON ((-96.63638 49.39646, -96.63597 49.396..."
2,310013155,310013151,0.000768,1301.543289,0.094106,59.975058,3028500.0,1.322273,0.100000,0.0,...,1301.543289,0.078864,0.000768,49.404947,-96.595011,0.004507,0.985797,Nelson,05OS,"MULTIPOLYGON (((-96.57027 49.42224, -96.56986 ..."
3,310013156,310013151,0.000496,6743.452378,0.136037,93.606178,18478800.0,3.324347,0.169820,0.0,...,11106.244584,0.095679,0.000661,49.404677,-96.595007,0.004507,0.985797,Nelson,05OS,"POLYGON ((-96.54527 49.37105, -96.54528 49.371..."
4,310013179,310013156,0.000032,796.690476,0.085525,40.444107,742500.0,1.155956,0.100000,0.0,...,796.690476,0.086501,0.000032,49.379474,-96.561250,0.004507,0.985797,Nelson,05OS,"POLYGON ((-96.56667 49.36742, -96.56708 49.367..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,310020587,310013155,-1.234500,-1.234500,0.068989,45.713966,4674600.0,1.033715,0.100000,0.0,...,-1.234500,0.068989,0.000670,49.410416,-96.583526,0.004507,0.985797,Nelson,05OS,"POLYGON ((-96.56681 49.40984, -96.56681 49.410..."
57,310020603,310013152,-1.234500,-1.234500,0.270868,122.177060,5176800.0,1.087038,0.100000,0.0,...,-1.234500,0.270868,0.002634,49.388680,-96.605107,0.004507,0.985797,Nelson,05OS,"POLYGON ((-96.59276 49.37511, -96.59279 49.377..."
58,310020623,310013179,-1.234500,-1.234500,0.086642,64.575319,5121900.0,1.081340,0.100000,0.0,...,-1.234500,0.086642,0.002213,49.376452,-96.570287,0.004507,0.985797,Nelson,05OS,"POLYGON ((-96.56341 49.33826, -96.56588 49.338..."
59,310020644,310013184,-1.234500,-1.234500,0.122003,89.093174,8779500.0,1.410327,0.101521,0.0,...,-1.234500,0.122003,0.003489,49.360158,-96.535850,0.004507,0.985797,Nelson,05OS,"POLYGON ((-96.52578 49.32739, -96.52578 49.327..."


In [167]:
# Calculate time of concentration and time to peak
cn = 63
mc_clrh_gdf_project['BasinsArea_acres'] = mc_clrh_gdf_project['BasArea'] * 0.00024710538
mc_clrh_gdf_project['FlowLength_ft'] = 209 * (mc_clrh_gdf_project['BasinsArea_acres']**0.6)
mc_clrh_gdf_project['BasSlope_prcnt'] = np.tan(np.radians(mc_clrh_gdf_project['BasSlope'])) * 100
mc_clrh_gdf_project['max_retention'] = (1000 / cn) - 10
mc_clrh_gdf_project['time_of_conc'] = ((mc_clrh_gdf_project['FlowLength_ft']**0.8) * ((mc_clrh_gdf_project['max_retention'] + 1)**0.7)) / (1140 * (mc_clrh_gdf_project['BasSlope_prcnt']**0.5))
mc_clrh_gdf_project['time_to_peak'] = mc_clrh_gdf_project['time_of_conc'] * 0.6

mc_clrh_gdf_project.head()

,SubId,DowSubId,RivSlope,RivLength,BasSlope,BasAspect,BasArea,BkfWidth,BkfDepth,Lake_Cat,...,c,SSDA_NM,SDA_NM,geometry,BasinsArea_acres,FlowLength_ft,BasSlope_prcnt,max_retention,time_of_conc,time_to_peak
0,310013151,310015352,0.001359,1472.132034,0.139288,101.631184,1020600.0,3.598586,0.178092,0.0,...,0.985797,Nelson,05OS,"POLYGON ((-96.6018 49.40315, -96.60346 49.4031...",252.195751,5770.174473,0.243105,5.873016,7.000884,4.200530
1,310013152,310015352,0.000777,3362.132034,0.341033,113.693816,5556600.0,1.557162,0.107737,0.0,...,0.985797,Nelson,05OS,"POLYGON ((-96.63638 49.39646, -96.63597 49.396...",1373.065755,15950.014693,0.595222,5.873016,10.091778,6.055067
2,310013155,310013151,0.000768,1301.543289,0.094106,59.975058,3028500.0,1.322273,0.100000,0.0,...,0.985797,Nelson,05OS,"MULTIPOLYGON (((-96.57027 49.42224, -96.56986 ...",748.358643,11081.847094,0.164246,5.873016,14.356245,8.613747
3,310013156,310013151,0.000496,6743.452378,0.136037,93.606178,18478800.0,3.324347,0.169820,0.0,...,0.985797,Nelson,05OS,"POLYGON ((-96.54527 49.37105, -96.54528 49.371...",4566.210896,32800.454815,0.237430,5.873016,28.446865,17.068119
4,310013179,310013156,0.000032,796.690476,0.085525,40.444107,742500.0,1.155956,0.100000,0.0,...,0.985797,Nelson,05OS,"POLYGON ((-96.56667 49.36742, -96.56708 49.367...",183.475745,4767.530515,0.149269,5.873016,7.669169,4.601501


In [168]:
# Create lists of necessary columns
mc_subids = list(mc_clrh_gdf_project['SubId'])
mc_rivlengths = list(mc_clrh_gdf_project['RivLength'])
mc_hasgauge = list(mc_clrh_gdf_project['Has_POI'])
mc_dowsubids = list(mc_clrh_gdf_project['DowSubId'])
mc_basarea = list(mc_clrh_gdf_project['BasArea'])
mc_elev = list(mc_clrh_gdf_project['MeanElev'])
mc_lat = list(mc_clrh_gdf_project['centroid_y'])
mc_lon = list(mc_clrh_gdf_project['centroid_x'])
mc_slope = list(mc_clrh_gdf_project['BasSlope'])
mc_aspect = list(mc_clrh_gdf_project['BasAspect'])
mc_bkfwidth = list(mc_clrh_gdf_project['BkfWidth'])
mc_bkfdepth = list(mc_clrh_gdf_project['BkfDepth'])
mc_rivslope = list(mc_clrh_gdf_project['RivSlope'])
mc_floodpn = list(mc_clrh_gdf_project['FloodP_n'])
mc_chn = list(mc_clrh_gdf_project['Ch_n'])
mc_timeofconc = list(mc_clrh_gdf_project['time_of_conc'])
mc_timetopeak = list(mc_clrh_gdf_project['time_to_peak'])

In [136]:
# Modify rivlengths
# Convert to km
mc_rivlengths_km = [x / 1000 for x in mc_rivlengths]
# Round to 3 decimals
mc_rivlengths_km_rounded = [ "{:0.3f}".format(v) for v in mc_rivlengths_km]
# Replace invalid river lengths with 'ZERO-'
mc_rivlengths_mod = list(map(lambda x: 'ZERO-' if x == '-0.001' else x, mc_rivlengths_km_rounded))
# Replace invalid river lengths with 0.00
mc_rivlengths_mod_n = list(map(lambda x: 0.0 if x == -0.0012345 else x, mc_rivlengths_km))

In [138]:
# Modify rivslope
mc_rivslope_mod = list(map(lambda x: 0.0001234 if x <= 0.0001 else x, mc_rivslope))

In [10]:
# Convert basin areas to km2
mc_basarea_km2 = [x / 1000000 for x in mc_basarea]

In [101]:
# Extract outlet subid and drainage area
mc_outlet = mc_clrh_gdf_project.loc[mc_clrh_gdf_project['Has_POI'] == 1, 'SubId'].item()
mc_outlet_da = mc_clrh_gdf_project.loc[mc_clrh_gdf_project['Has_POI'] == 1, 'DrainArea'].item()
# Convert drainage area to km2
mc_outlet_da_km2 = mc_outlet_da / 1000000

In [112]:
# Replace invalid bankfull depth valyes with 1
mc_bkfdepth_mod = list(map(lambda x: 1 if x < 1 else x, mc_bkfdepth))

## WSC RVT File

In [ ]:
# Read in wsc metadata file
wsc_metadata_gdf = gpd.read_file('/Users/emilito/Documents/PWA/Data/Raw/manning/WSC_data/station_metadata_historical.csv')

# Prompt user to input file name of WSC historic data
wsc_data_file_name = input('WSC data file name (e.g., daily_20250327T1926): ')

# Clean WSC data by removing first row
with open('/Users/emilito/Documents/PWA/Data/Raw/manning/WSC_data/' + wsc_data_file_name + '.csv', 'r') as f, open('/Users/emilito/Documents/PWA/Data/Raw/manning/WSC_data/' + wsc_data_file_name + '_clean.csv', 'w') as f1:
    next(f) # skip header line
    for line in f:
        f1.write(line)

# Create WSC dataframe
wsc_data_df = pd.read_csv('/Users/emilito/Documents/PWA/Data/Raw/manning/WSC_data/' + wsc_data_file_name + '_clean.csv')

# Extract discharge values from dataframe and replace NaN values with -1.2345
wsc_data_dschrg = wsc_data_df.loc[wsc_data_df['PARAM'] == 1, 'Value']
wsc_data_dschrg.fillna(-1.2345, inplace = True)

# Extract start and end date from WSC dataframe
start_date = datetime.strptime(wsc_data_df['Date'][0], '%Y/%m/%d').strftime("%Y-%m-%d")
end_date = datetime.strptime(wsc_data_df['Date'][len(wsc_data_df) - 1], '%Y/%m/%d').strftime("%Y-%m-%d")

# Create file and store in raw data folder
file_path = '/Users/emilito/Documents/PWA/Data/Raw/manning/Raven_Input_Files/WSC' + wsc_metadata_gdf['Station ID'].values[0] + '.rvt'

# Open file in write mode
with open(file_path, 'w') as file:
    # Write header content to file
    file.write(f"{'#################################################################################\n# Name = '}{wsc_metadata_gdf['Station Name'].values[0]}{'\n# ID = '}\
{wsc_metadata_gdf['Station ID'].values[0]}{'\n# Drainage Area = '}{int(mc_outlet_da_km2)}{' km2\n# Sub-basin ID = '}{mc_outlet}{'\n# Natural/Reference = Regulated\n# Country = CA\n# Date Range = '}\
{start_date}{' - '}{end_date}{'\n# Data time-step = 24 hr\n# Missing data are filled with -1.2345\n##################################################################################\
\n:ObservationData	HYDROGRAPH	'}{mc_outlet}{'	m3/s\n	'}{datetime.strptime(wsc_data_df['Date'][0], '%Y/%m/%d')}{'	1	'}{len(wsc_data_dschrg)}{'\n'}")

# Open file in append mode
with open(file_path, 'a') as file:
    # Append WSC data values to file
    for i in range(len(wsc_data_dschrg)):
        file.write(f"{'	'}{format(wsc_data_dschrg[i], '.4f')}{'\n'}")
    file.write(':EndObservationData')

## RVI File

In [ ]:
# Create file and store in raw data folder
file_path = DATA_RAW + 'Raven_Input_Files/' + model_name.replace(" ", "") + '.rvi'

# Open file in write mode
with open(file_path, 'w') as file:
    # Write model setup info to file (\ added for readability)
    file.write('# ----------------------------------------------\n# Raven Input file\n# ' + model_name + '\n# ----------------------------------------------\
\n:StartDate\
\n:EndDate\
\n:Method                ORDERED_SERIES                # Numerical method used for simulation\
\n:TimeStep              01:00:00                      # Expressed in days as a real-valued number (e.g., 1.0 for one day) or using hh:mm:ss format (e.g., 24:00:00 for one hour).\
\n:RunName               ManningCanal                  # The name of model run. It will be added as a prefix of model output files.\
\n\
\n:CatchmentRoute        ROUTE_TRI_CONVOLUTION         # Catchment routing method, used to convey water from the catchment tributaries and rivulets to the subbasin outlets. DEFAULT ROUTE_DUMP, which instantly ‘dumps’ all water in the subbasin stream reach.\
\n:Routing               ROUTE_DIFFUSIVE_WAVE          # Channel routing method which is used to transport water from upstream to downstream within the main subbasin channels. DEFAULT ROUTE_DIFFUSIVE_WAVE, which analytically solves the diffusive wave equation along the reach using a constant reference celerity.\
\n:PrecipIceptFract      PRECIP_ICEPT_NONE             # Estimation of the precipitation interception fraction. In this routing model, stream input(s) are "pretending" to be precipitation going into Raven, thus using DEFAULT PRECIP_ICEPT_NONE to indicate no interception processes are adopted.\
\n:PotentialMeltMethod   POTMELT_NONE                  # Estimation of the potential snow melt. In this routing model, snow melt processes are not relevant, thus using DEFAULT POTMELT_NONE method.\
\n:SWRadiationMethod     SW_RAD_NONE                   # Estimation of shortwave incoming radiation. This is turned off for a routing-only model.\
\n:SoilModel             SOIL_ONE_LAYER                # In this routing model, use DEFAULT SOIL_ONE_LAYER to define single soil layer structure.\
\n\
\n:HydrologicProcesses\
\n  :Precipitation     PRECIP_RAVEN             ATMOS_PRECIP     PONDED_WATER          # Moves stream input(s) from ATMOS_PRECIP to PONDED_WATER storage (waiting for runoff). Use DEFAULT PRECIP_RAVEN method.\
\n  :Abstraction	   ABST_FILL		        PONDED_WATER     DEPRESSION\
\n  :Flush             RAVEN_DEFAULT            PONDED_WATER     SURFACE_WATER         # Moves water from PONDED_WATER to SURFACE_WATER (routed to outlet). Use DEFAULT RAVEN_DEFAULT method.\
\n  :Seepage 	       SEEP_LINEAR 	            DEPRESSION       SOIL[0]\
\n:EndHydrologicProcesses\
\n\
\n# Defines the hydrograph performance metrics output by Raven. Either one or multiple is acceptable.\
\n:EvaluationMetrics			NASH_SUTCLIFFE    PCT_BIAS    KLING_GUPTA')

In [99]:
# Obtain model start and end dates from user input
start_date = input("start date (YYYY-MM-DD HH:MM:SS): ")
end_date = input("end date (YYYY-MM-DD HH:MM:SS): ")

In [100]:
# Add start and end dates defined by user to .rvi file
file_name = DATA_RAW + 'Raven_Input_Files/ManningCanal.rvi'
for line in fileinput.FileInput(file_name, inplace=1):
    if ':StartDate' in line:
        line = line.rstrip()
        line = line.replace(line, line + '             ' + start_date + '           # Model run start time, format in yyyy-mm-dd hh:mm:ss' + '\n')
    if ':EndDate' in line:
        line = line.rstrip()
        line = line.replace(line, line + '               ' + end_date + '           # Model run end time, format in yyyy-mm-dd hh:mm:ss' + '\n')
    print(line, end="")

## RVC File

In [107]:
# Create file and store in raw data folder
file_path = DATA_RAW + 'Raven_Input_Files/' + model_name.replace(" ", "") + '.rvc'

# Open file in write mode
with open(file_path, 'w') as file:
    # Write initial conditions to file
    file.write('# ----------------------------------------------\n# Raven Input file\n# ' + model_name + '\n# ----------------------------------------------\n\
\n:UniformInitialConditions SOIL[0] 15            # Initial conditions for the groundwater storage, in mm. DEFAULT 15 mm.')

## RVP File

In [190]:
# Create file and store in raw data folder
file_path = DATA_RAW + 'Raven_Input_Files/' + model_name.replace(" ", "") + '.rvp'

# Open file in write mode
with open(file_path, 'w') as file:
    # Write soil, vegetation, and landuse content to file (\ added for readability)
    file.write(f"{'# ----------------------------------------------\n# Raven Input file\n# ' + model_name + '\n# ----------------------------------------------\
\n\
\n# Soil/Vegetation/Land use class definitions ------\
\n:SoilClasses\
\n  AQUIFER      # only one soil class used here\
\n:EndSoilClasses\
\n\
\n:SoilProfiles\
\n  # name, number of layers, soil class, thickness [m]\
\n  LandHRU_soil_profile,    '}{1}{',    AQUIFER, '}{5}\
{'\n:EndSoilProfiles\
\n\
\n# the vegetation and land use parameters unused in routing\
\n:VegetationClasses\
\n  :Attributes,                MAX_HT,       MAX_LAI,    MAX_LEAF_COND\
\n  :Units,                       m,            none,       mm_per_s\
\n  LandHRU_vege_class,           '}{25}{',           '}{5.0}{',          '}{5.0}\
{'\n:EndVegetationClasses\
\n\
\n:LandUseClasses\
\n  :Attributes,        IMPERMEABLE_FRAC,         FOREST_COVERAGE\
\n  :Units,                     fract,                    fract\
\n  LandHRU_landuse_class,      '}{0}{',                        '}{1}")
    
# Open file in append mode
with open(file_path, 'a') as file:
    # Append land use classes and parameters to file
    for i in range(len(mc_subids)):
        file.write(f"{'\n  SUB'}{str(mc_subids[i])}{',		      '}{0}{',                        '}{1}")
    file.write(f"{'\n:EndLandUseClasses\
\n\
\n# Global Parameters-----------------------------\
\n:AvgAnnualRunoff 		'}{75}\
{'\n'}")
    file.write(f"{'\n:LandUseParameterList'}{'\n  :Parameters' : <12}{',' : <1}{'DEP_SEEP_K,' : >20}{'DEP_MAX' : >20}{'\n  :Units' : <14}{',' : <1}{'1/d,' : >20}{'mm' : >20}\
{'\n    [DEFAULT]' : <14}{',' : <1}{0.01 : >21}{',' : <1}{'20.0000' : >18}")
    for i in range(len(mc_subids)):
        file.write(f"{'\n SUB' + str(mc_subids[i]) : <14}{',' : <1}{0.01 : >21}{',' : <1}")
    file.write('\n:EndLandUseClasses\n\
\n# redirects to channel properties file\
\n:RedirectToFile         channel_properties.rvp\n')

## RVT File

In [108]:
# Prompt user for PHyDAP NetCDF file name (file generated from netcdf_add_dimension.R)
nc_file_name = input("NetCDF file name with extension (.nc): ")

In [109]:
# Create file and store in raw data folder
file_path = DATA_RAW + 'Raven_Input_Files/' + model_name.replace(" ", "") + '.rvt'

# Open file in write mode
with open(file_path, 'w') as file:
    # Write PHyDAP netcdf metadata to file and point to GridWeights and WSC files (\ added for readability)
    file.write('# ----------------------------------------------\n# Raven Input file\n# ' + model_name + '\n# ----------------------------------------------\
\n\
\n:GriddedForcing           StreamInputs\
\n    :ForcingType          PRECIP\
\n    :FileNameNC           input/' + nc_file_name + '\
\n    :VarNameNC            runoff\
\n    :DimNamesNC           lon lat time     # must be in the order of (x,y,t)\
\n    :LinearTransform      24.0     0.0\
\n    :RedirectToFile       input/GridWeights.txt\
\n:EndGriddedForcing\
\n# NOTE: the Linear Transform converts mm/1hr --> mm/d\
\n\
\n:GriddedForcing             Averagetemp\
\n    :ForcingType            TEMP_AVE\
\n    :FileNameNC             input/' + nc_file_name + '\
\n    :VarNameNC              runoff\
\n    :DimNamesNC             lon lat time\
\n    :RedirectToFile         input/GridWeights.txt\
\n:EndGriddedForcing\
\n\
\n# Observed Discharge Data-----------------------\
\n:RedirectToFile	            WSC' + wsc_metadata_gdf['Station ID'].values[0] + '.rvt')

## RVH File

In [172]:
# Create file and store in raw data folder
file_path = DATA_RAW + 'Raven_Input_Files/' + model_name.replace(" ", "") + '.rvh'

# Open file in write mode
with open(file_path, 'w') as file:
    # Write SubBasin metadata to file (\ added for readability)
    file.write('# ----------------------------------------------\n# Raven Input file\n# ' + model_name + '\n# ----------------------------------------------\n\
\n## SubBasin List---------------------------------------------------------------------\
\n# ID:            A positive long integer identifier unique to this subbasin.\
\n# NAME:          A nickname for subbasin. DEFAULT string("sub_" + ID).\
\n# DOWNSTREAM_ID: The ID of the subbasin that receives this subbasins outflowing waters.\
\n# PROFILE:       The representative channel profile code (channel profiles specified in the channel_properties.rvp file). DEFAULT string("chn_" + ID)\
\n# REACH_LENGTH:  The length of the primary reach channel in the basin (in km).\
\n# GAUGED:        Flag which determines whether modeled hydrographs for this subbasin are generated as output from the model (either 1 or 0, true or false). In this \
routing model, both the subbasins with reservoir and flow gauge are set as GAUGED = 1.\
\n#\
\n:SubBasins\
\n  :Attributes ID  NAME  DOWNSTREAM_ID PROFILE REACH_LENGTH  GAUGED\
\n  :Units    none  none           none    none           km    none\n')

# Open file in append mode
with open(file_path, 'a') as file:
    # Append SubBasin data to file
    for i in range(len(mc_subids)):
        file.write(f"{str(mc_subids[i]) : >15}{'sub' + str(mc_subids[i]) : >20}{str(mc_dowsubids[i]) : >15}{'chn_' + str(mc_subids[i]) : >20}{mc_rivlengths_mod[i] : >20}{mc_hasgauge[i] : >10}{'\n'}")#{mc_rivlengths_km[i], : >20}


# Open file in append mode
with open(file_path, 'a') as file:
    # Append HRU metadata to file (\ added for readability)
    file.write(':EndSubBasins\n\
\n## HRU List---------------------------------------------------------------------\
\n# HRU ID:          A positive integer unique to this HRU. Note that More than one HRUs may have the same subbasin ID, but HRU ID is unique.\
\n# AREA:            HRU area value, in km2.\
\n# ELEVATION:       HRU mean elevation, in m.\
\n# LATITUDE:        Lattitude of HRU centroid, in decimal degrees.\
\n# LONGITUDE:       longitude of HRU centroid, in decimal degrees.\
\n# BASIN_ID:        The ID of the subbasin in which the HRU is located (as defined in the :SubBasins command).\
\n# LAND_USE_CLASS:  Land use classes defined in Lievre.rvp file. DEFAULT LandHRU_landuse_class or LakeHRU_landuse_class for land and lake HRU, respectively.\
\n# VEG_CLASS:       Vegetation classes defined in Lievre.rvp file. DEFAULT LandHRU_vege_class or LakeHRU_vege_class for land and lake HRU, respectively.\
\n# SOIL_PROFILE:    Soil profile defined in Lievre.rvp file. DEFAULT LandHRU_soil_profile or LakeHRU_soil_profile for land and lake HRU, respectively.\
\n# AQUIFER_PROFILE: Aquifer classes defined in Lievre.rvp file. DEFAULT [NONE] in this routing model.\
\n# TERRAIN_CLASS:   Terrain classes defined in Lievre.rvp file. DEFAULT [NONE] in this routing model.\
\n# SLOPE:           HRU mean slope, in degrees.\
\n# ASPECT:          HRU mean aspect, in degrees. Northern:0°, western: 90°, southern: 180°, eastern: 270°.\
\n#\
\n:HRUs\
\n  :Attributes      AREA  ELEVATION       LATITUDE      LONGITUDE BASIN_ID                 LAND_USE_CLASS                      VEG_CLASS      SOIL_PROFILE  AQUIFER_PROFILE TERRAIN_CLASS      SLOPE     ASPECT\
\n  :Units            km2          m            deg            deg     none                           none                           none                           none     none     none        deg       degN\n')
    

# Open file in append mode
with open(file_path, 'a') as file:
    # Append HRU data to file
    for i in range(len(mc_subids)):
        file.write(f"{str(mc_subids[i]) : >15}{round(mc_basarea_km2[i], 4) : >10}{round(mc_elev[i], 2) : >10}{round(mc_lat[i], 6) : >15}{round(mc_lon[i], 6) : >15}{str(mc_subids[i]) : >15}{'LandHRU_landuse_class' : >30}{'LandHRU_vege_class' : >25}{'LandHRU_soil_profile' : >30}{'[NONE]' : >10}{'[NONE]' : >10}{round(mc_slope[i], 2) : >10}{round(mc_aspect[i], 2) : >10}{'\n'}")
    file.write(':EndHRUs\n')

# Open file in append mode
with open(file_path, 'a') as file:
    # Append time of conc and time to peak data to file
    file.write('\n# SubBasin Properties ----------------------------------------------------\n:SubBasinProperties\n  :Parameters	TIME_CONC  TIME_TO_PEAK\n  :Units              hrs           hrs\n')
    for i in range(len(mc_subids)):
        file.write(f"{str(mc_subids[i]) : >15}{round(mc_timeofconc[i], 2) : >10}{round(mc_timetopeak[i], 2) : >14}{'\n'}")
    file.write(':EndSubBasinProperties')

## Channel Properties RVP File

In [139]:
# Create file and store in raw data folder
file_path = DATA_RAW + 'Raven_Input_Files/channel_properties.rvp'

# Open file in write mode
with open(file_path, 'w') as file:
    # Write header content to file
    file.write('# ----------------------------------------------\n# Raven Input file\n# ' + model_name + '\n# ----------------------------------------------\n')

# Open file in append mode      
with open(file_path, 'a') as file:
    # Write channel transect data for each subbasin to file
    for i in range(len(mc_subids)):
        # Define survey point x value equations
        x1 = 0
        x2 = mc_bkfdepth_mod[i] * 2
        x3 = (mc_bkfwidth[i] - (mc_bkfdepth_mod[i]*3))*2
        x4 = x2 + x3
        x5 = (mc_bkfwidth[i] - (mc_bkfdepth_mod[i]*2)) + x3
        x6 = x5 + x2
        x7 = x3 - x2 + x6
        x8 = x7 + x2
        file.write(f"{'\n###################################chn_'}{str(mc_subids[i])}{'\n##Subbasin ID:        '}{str(mc_subids[i])}{'\n##Subbasin area:      '}{round(mc_basarea_km2[i], 2)}{' km2\n\
##Channel length:     '}{round(mc_rivlengths_mod_n[i], 2)}{' km\n##Channel elevation:  '}{round(mc_elev[i], 3)}{' m\n##Bankfull width:     '}{round(mc_bkfwidth[i], 4)}{' m\n##Bankfull depth:     '}\
{round(mc_bkfdepth_mod[i], 2)}{' m\n:ChannelProfile			chn_'}{str(mc_subids[i])}{'\n        :Bedslope			 '}{round(mc_rivslope_mod[i], 7)}{'\n        :SurveyPoints\n		   '}{0 : >5}{'  '}\
{round(mc_elev[i] + mc_bkfdepth_mod[i] + (mc_bkfdepth_mod[i]/2), 2)}{'\n		   '}{format(x2, '.2f') : >5}{'  '}{round(mc_elev[i] + mc_bkfdepth_mod[i], 2)}{'\n		   '}{format(x3, '.2f') : >5}\
{'  '}{round(mc_elev[i] + mc_bkfdepth_mod[i], 2)}{'\n		   '}{format(x4, '.2f') : >5}{'  '}{round(mc_elev[i], 2)}{'\n		   '}{format(x5, '.2f') : >5}{'  '}{round(mc_elev[i], 2)}{'\n		   '}\
{format(x6, '.2f') : >5}{'  '}{round(mc_elev[i] + mc_bkfdepth_mod[i], 2)}{'\n		   '}{format(x7, '.2f') : >5}{'  '}{round(mc_elev[i] + mc_bkfdepth_mod[i], 2)}{'\n		   '}{format(x8, '.2f') : >5}\
{'  '}{round(mc_elev[i] + mc_bkfdepth_mod[i] + (mc_bkfdepth_mod[0]/2), 2)}{'\n	:EndSurveyPoints\n	:RoughnessZones\n		   '}{0 : >5}{' '}{format(mc_floodpn[i], '.6f')}{'\n		   '}{format(x3, '.2f') : >5}\
{' '}{format(mc_chn[i], '.6f')}{'\n		   '}{format(x6, '.2f') : >5}{' '}{format(mc_floodpn[i], '.6f')}{'\n	:EndRoughnessZones\n:EndChannelProfile\n'}")

## GridWeights TXT File

In [196]:
# Create file and store in raw data folder
file_path = DATA_RAW + 'Raven_Input_Files/GridWeights.txt'

# Open file in write mode
with open(file_path, 'w') as file:
    # Write header and metadata to file
    file.write(f"{'# ----------------------------------------------\n# Raven Input file\n# ' + model_name + '\n# ----------------------------------------------\n\
\n:GridWeights\
\n   #\
\n   # [# HRUs]\
\n   :NumberHRUs       '}{len(mc_subids)}\
{'\n   :NumberGridCells  '}{4175}\
{'\n   #'}\
{'\n   # [HRU ID] [Cell #] [w_kl]'}")

# Open file in append mode
with open(file_path, 'a') as file:
    # Append grid weights data to file
    for i in range(len(mc_subids)):
        file.write(f"{'\n  '}{str(mc_subids[i]) : >10}{108 : >10}{1.0 : >10}")
    file.write("\n:EndGridWeights")